In [21]:
import os
import re
import csv
import json
import math
import time
import random
import zipfile
import collections
from pathlib import Path
from contextlib import nullcontext

import numpy as np
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.transforms import InterpolationMode
from tqdm.auto import tqdm

try:
    from IPython.display import display
except Exception:
    display = None

# speed knobs, safe for conv-heavy training
torch.backends.cudnn.benchmark = True
if hasattr(torch.backends.cuda.matmul, "allow_tf32"):
    torch.backends.cuda.matmul.allow_tf32 = True
if hasattr(torch.backends.cudnn, "allow_tf32"):
    torch.backends.cudnn.allow_tf32 = True

class CFG:
    # leave as None for auto-detection under /kaggle/input
    DATA_ROOT = Path("AIC-UIT/visual-question-answering/datasets")

    # image size keeps the CLEVR wide aspect ratio instead of squeezing everything square
    IMG_H = 160
    IMG_W = 240

    MAX_LEN = 48
    MIN_WORD_FREQ = 1

    # increase EPOCHS / SEEDS if the GPU time budget allows
    EPOCHS = 28
    BATCH_SIZE = 64
    LR = 2.5e-4
    WEIGHT_DECAY = 2e-4
    LABEL_SMOOTHING = 0.03
    DROPOUT = 0.20
    GRAD_CLIP = 2.0
    NUM_WORKERS = min(4, os.cpu_count() or 2)
    AMP = True
    COMPILE = False  # torch.compile can be faster, but Kaggle sometimes dislikes it

    # two different validation splits + averaged logits usually beats one model
    # set to [3407, 2025, 777] if time allows
    SEEDS = [3407, 2025, 777]
    VAL_RATIO = 0.05

    # CLEVR/CLOSURE answers are type-constrained. this is a huge accuracy stabilizer
    # "hard" = impossible answers get -inf. Use "soft" if val score says hard is too strict
    ANSWER_MASK_MODE = "hard"   # one of: "none", "soft", "hard"
    ANSWER_MASK_BOOST = 3.0

    OUT_DIR = Path("outputs")
    MODEL_DIR = OUT_DIR / "models"
    SUBMISSION_CSV = OUT_DIR / "submission.csv"
    SUBMISSION_ZIP = OUT_DIR / "submission.zip"

CFG.OUT_DIR.mkdir(parents=True, exist_ok=True)
CFG.MODEL_DIR.mkdir(parents=True, exist_ok=True)

print("Output dir:", CFG.OUT_DIR.resolve())
print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")


Output dir: /home/izu/Projects/AIC-UIT/visual-question-answering/outputs
CUDA: True NVIDIA GeForce RTX 4050 Laptop GPU


In [22]:
def seed_everything(seed: int) -> None:
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def find_dataset_root() -> Path:
    """Find a folder containing questions/train.json and images/train."""
    candidates = []
    if CFG.DATA_ROOT is not None:
        candidates.append(Path(CFG.DATA_ROOT))

    candidates += [
        Path("/kaggle/input/synvqa-uitaic"),
        Path("/kaggle/input/datasets/notnguyen/synvqa-uitaic"),
        Path("/kaggle/input"),
        Path("."),
    ]

    def ok(p: Path) -> bool:
        return (p / "questions" / "train.json").exists() and (p / "images" / "train").exists()

    for p in candidates:
        if ok(p):
            return p

    # full recursive search under Kaggle input
    for base in [Path("/kaggle/input"), Path(".")]:
        if not base.exists():
            continue

        for root, dirs, files in os.walk(base):
            p = Path(root)
            if ok(p):
                return p

    raise FileNotFoundError("Could not find dataset root with questions/train.json and images/train.")


def read_questions(path: Path):
    with open(path, "r", encoding="utf-8") as f:
        obj = json.load(f)

    if isinstance(obj, dict):
        if "questions" in obj:
            return obj["questions"]
        if "data" in obj:
            return obj["data"]

    if isinstance(obj, list):
        return obj

    raise ValueError(f"Unknown JSON schema in {path}")


def get_question(item) -> str:
    return str(item.get("question", ""))


def get_answer(item) -> str:
    return str(item.get("answer", "")).strip().lower()


def get_qid(item, fallback_idx=None) -> str:
    for k in ["qid", "question_id", "id"]:
        if k in item:
            return str(item[k])

    return str(fallback_idx)


def get_image_filename(item) -> str:
    for k in ["image_filename", "image", "filename", "file_name"]:
        if k in item:
            return str(item[k])

    raise KeyError("Item does not contain image_filename/image/filename/file_name")


def guess_domain(item) -> str:
    s = (get_qid(item, "") + " " + get_image_filename(item)).lower()
    if "closure" in s:
        return "closure"
    if "clevr" in s:
        return "clevr"

    return "unknown"

root = find_dataset_root()
TRAIN_JSON = root / "questions" / "train.json"
TEST_JSON  = root / "questions" / "test.json"
TRAIN_IMG_DIR = root / "images" / "train"
TEST_IMG_DIR  = root / "images" / "test"

train_items_all = read_questions(TRAIN_JSON)
test_items = read_questions(TEST_JSON)

print("Dataset root:", root)
print("Train questions:", len(train_items_all), " Test questions:", len(test_items))
print("Train image dir:", TRAIN_IMG_DIR)
print("Test image dir:", TEST_IMG_DIR)
print("Sample item:", train_items_all[0])


Dataset root: datasets
Train questions: 699989  Test questions: 227899
Train image dir: datasets/images/train
Test image dir: datasets/images/test
Sample item: {'image_index': 0, 'program': [{'inputs': [], 'function': 'scene', 'value_inputs': []}, {'inputs': [0], 'function': 'filter_size', 'value_inputs': ['large']}, {'inputs': [1], 'function': 'filter_color', 'value_inputs': ['green']}, {'inputs': [2], 'function': 'count', 'value_inputs': []}, {'inputs': [], 'function': 'scene', 'value_inputs': []}, {'inputs': [4], 'function': 'filter_size', 'value_inputs': ['large']}, {'inputs': [5], 'function': 'filter_color', 'value_inputs': ['purple']}, {'inputs': [6], 'function': 'filter_material', 'value_inputs': ['metal']}, {'inputs': [7], 'function': 'filter_shape', 'value_inputs': ['cube']}, {'inputs': [8], 'function': 'count', 'value_inputs': []}, {'inputs': [3, 9], 'function': 'greater_than', 'value_inputs': []}], 'question_index': 0, 'image_filename': 'CLEVR_train_000000.png', 'question_fa

In [23]:
TOKEN_RE = re.compile(r"[a-z0-9]+")

def tokenize(text: str):
    return TOKEN_RE.findall(text.lower())


def build_vocab(items):
    counter = collections.Counter()
    for item in items:
        counter.update(tokenize(get_question(item)))

    vocab = {"<PAD>": 0, "<UNK>": 1}
    for word, count in counter.most_common():
        if count >= CFG.MIN_WORD_FREQ and word not in vocab:
            vocab[word] = len(vocab)

    return vocab


def encode_question(text: str, vocab: dict):
    ids = [vocab.get(tok, vocab["<UNK>"]) for tok in tokenize(text)]
    ids = ids[:CFG.MAX_LEN]

    if len(ids) < CFG.MAX_LEN:
        ids += [vocab["<PAD>"]] * (CFG.MAX_LEN - len(ids))

    return torch.tensor(ids, dtype=torch.long)


def build_answer_vocab(items):
    answers = sorted({get_answer(item) for item in items if "answer" in item})
    ans2idx = {a: i for i, a in enumerate(answers)}
    idx2ans = answers

    return ans2idx, idx2ans

vocab = build_vocab(train_items_all)
ans2idx, idx2ans = build_answer_vocab(train_items_all)

print("Vocab size:", len(vocab))
print("Answer classes:", len(idx2ans), idx2ans[:50])

with open(CFG.OUT_DIR / "vocab.json", "w", encoding="utf-8") as f:
    json.dump(vocab, f, ensure_ascii=False, indent=2)
with open(CFG.OUT_DIR / "answers.json", "w", encoding="utf-8") as f:
    json.dump(idx2ans, f, ensure_ascii=False, indent=2)


Vocab size: 82
Answer classes: 28 ['0', '1', '10', '2', '3', '4', '5', '6', '7', '8', '9', 'blue', 'brown', 'cube', 'cyan', 'cylinder', 'gray', 'green', 'large', 'metal', 'no', 'purple', 'red', 'rubber', 'small', 'sphere', 'yellow', 'yes']


In [24]:
def split_by_image(items, seed: int, val_ratio: float):
    """Group by image so validation does not leak the same scene from train."""
    if val_ratio <= 0:
        return items, []

    image_to_items = collections.defaultdict(list)
    image_to_domain = {}

    for item in items:
        fn = get_image_filename(item)
        image_to_items[fn].append(item)
        image_to_domain[fn] = guess_domain(item)

    rng = random.Random(seed)
    domain_to_images = collections.defaultdict(list)

    for fn, dom in image_to_domain.items():
        domain_to_images[dom].append(fn)

    val_images = set()

    for dom, imgs in domain_to_images.items():
        imgs = imgs[:]
        rng.shuffle(imgs)
        n_val = max(1, int(round(len(imgs) * val_ratio))) if len(imgs) > 1 else 0
        val_images.update(imgs[:n_val])

    train_items, val_items = [], []

    for fn, group in image_to_items.items():
        if fn in val_images:
            val_items.extend(group)
        else:
            train_items.extend(group)

    return train_items, val_items

# preview split quality for seed 0.
tr_preview, va_preview = split_by_image(train_items_all, CFG.SEEDS[0], CFG.VAL_RATIO)

print("Preview split:", len(tr_preview), "train /", len(va_preview), "val")
print("Val domains:", collections.Counter(guess_domain(x) for x in va_preview))
print("Val answers top:", collections.Counter(get_answer(x) for x in va_preview).most_common(10))


Preview split: 664989 train / 35000 val
Val domains: Counter({'clevr': 35000})
Val answers top: [('no', 7117), ('yes', 7014), ('1', 2890), ('0', 2400), ('metal', 1610), ('small', 1578), ('rubber', 1564), ('large', 1539), ('2', 1415), ('cube', 1149)]


In [25]:
def make_transforms(train: bool):
    if train:
        # do not use horizontal flip: left/right questions would become wrong
        return transforms.Compose([
            transforms.Resize((CFG.IMG_H, CFG.IMG_W), interpolation=InterpolationMode.BICUBIC),
            transforms.RandomAffine(degrees=0, translate=(0.015, 0.015), scale=(0.98, 1.02), fill=128),
            transforms.ColorJitter(brightness=0.06, contrast=0.06),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
        ])
    return transforms.Compose([
        transforms.Resize((CFG.IMG_H, CFG.IMG_W), interpolation=InterpolationMode.BICUBIC),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
    ])


def resolve_image_path(img_dir: Path, filename: str) -> Path:
    p = img_dir / filename
    if p.exists():
        return p

    # fallbacks for odd JSON paths
    p = img_dir / Path(filename).name
    if p.exists():
        return p

    stem = Path(filename).stem

    for ext in [".png", ".jpg", ".jpeg", ".webp"]:
        p = img_dir / f"{stem}{ext}"
        if p.exists():
            return p

    raise FileNotFoundError(f"Image not found for {filename} under {img_dir}")


class SynVQADataset(Dataset):
    def __init__(self, items, img_dir: Path, vocab: dict, ans2idx: dict | None, train: bool, return_raw: bool = False):
        self.items = list(items)
        self.img_dir = Path(img_dir)
        self.vocab = vocab
        self.ans2idx = ans2idx
        self.train = train
        self.return_raw = return_raw
        self.tfms = make_transforms(train)

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        item = self.items[idx]
        question = get_question(item)
        qid = get_qid(item, idx)
        img_path = resolve_image_path(self.img_dir, get_image_filename(item))
        img = Image.open(img_path).convert("RGB")
        img = self.tfms(img)
        tokens = encode_question(question, self.vocab)

        if self.train:
            answer = get_answer(item)
            label = torch.tensor(self.ans2idx[answer], dtype=torch.long)

            if self.return_raw:
                return img, tokens, label, qid, question

            return img, tokens, label

        return img, tokens, qid, question


def make_loader(ds, batch_size, shuffle):
    kwargs = dict(
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=CFG.NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
        drop_last=False,
    )
    if CFG.NUM_WORKERS > 0:
        kwargs["persistent_workers"] = True
        kwargs["prefetch_factor"] = 2

    return DataLoader(ds, **kwargs)


In [26]:
ANSWER_GROUP_WORDS = {
    "yesno": {"yes", "no"},
    "number": {str(i) for i in range(0, 21)} | {"zero", "one", "two", "three", "four", "five", "six", "seven", "eight", "nine", "ten"},
    "color": {"gray", "grey", "red", "blue", "green", "brown", "purple", "cyan", "yellow"},
    "shape": {"cube", "sphere", "cylinder", "cone", "torus"},
    "material": {"rubber", "metal", "matte", "shiny"},
    "size": {"small", "large", "big", "tiny"},
}


def build_answer_groups(idx2ans):
    groups = {}
    for group, words in ANSWER_GROUP_WORDS.items():
        ids = [i for i, a in enumerate(idx2ans) if str(a).lower() in words]
        if ids:
            groups[group] = torch.tensor(ids, dtype=torch.long)

    return groups


def infer_question_group(question: str):
    q = " ".join(tokenize(question))
    if not q:
        return None

    # attribute queries first
    if "how many" in q or "number of" in q:
        return "number"
    if "what color" in q or "color of" in q or "colour" in q:
        return "color"
    if "what material" in q or "made of" in q:
        return "material"
    if "what shape" in q or "shape is" in q or "what type of object" in q:
        return "shape"
    if "what size" in q or "size is" in q:
        return "size"

    # boolean / comparison questions
    first = q.split()[0]
    if first in {"is", "are", "was", "were", "do", "does", "did", "has", "have", "can"}:
        return "yesno"
    if any(x in q for x in ["same", "different", "equal", "greater than", "less than", "fewer than", "more than", "there any"]):
        return "yesno"

    return None


answer_groups_cpu = build_answer_groups(idx2ans)
print({k: [idx2ans[i] for i in v.tolist()] for k, v in answer_groups_cpu.items()})


def apply_answer_mask(logits: torch.Tensor, questions, answer_groups, mode: str = None):
    mode = CFG.ANSWER_MASK_MODE if mode is None else mode
    if mode == "none" or not answer_groups:
        return logits

    logits = logits.float()
    out = logits.clone()

    for i, q in enumerate(questions):
        g = infer_question_group(q)

        if g is None or g not in answer_groups:
            continue

        ids = answer_groups[g].to(logits.device)

        if mode == "hard":
            masked = torch.full_like(out[i], -1e9)
            masked[ids] = out[i, ids]
            out[i] = masked

        elif mode == "soft":
            out[i, ids] += CFG.ANSWER_MASK_BOOST

    return out


{'yesno': ['no', 'yes'], 'number': ['0', '1', '10', '2', '3', '4', '5', '6', '7', '8', '9'], 'color': ['blue', 'brown', 'cyan', 'gray', 'green', 'purple', 'red', 'yellow'], 'shape': ['cube', 'cylinder', 'sphere'], 'material': ['metal', 'rubber'], 'size': ['large', 'small']}


In [27]:
class ConvBNAct(nn.Module):
    def __init__(self, in_ch, out_ch, kernel=3, stride=1, padding=None, groups=1):
        super().__init__()

        if padding is None:
            padding = kernel // 2

        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel, stride=stride, padding=padding, groups=groups, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.SiLU(inplace=True),
        )
    def forward(self, x):
        return self.net(x)


class ResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.conv1 = ConvBNAct(in_ch, out_ch, 3, stride=stride)
        self.conv2 = nn.Sequential(
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
        )
        self.skip = nn.Identity() if (in_ch == out_ch and stride == 1) else nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 1, stride=stride, bias=False),
            nn.BatchNorm2d(out_ch),
        )
        self.act = nn.SiLU(inplace=True)

    def forward(self, x):
        return self.act(self.conv2(self.conv1(x)) + self.skip(x))


class ImageEncoder(nn.Module):
    def __init__(self, out_ch=256):
        super().__init__()
        self.net = nn.Sequential(
            ConvBNAct(3, 48, 5, stride=2, padding=2),      # H/2, W/2
            ResBlock(48, 64, stride=1),
            ResBlock(64, 96, stride=2),                    # H/4, W/4
            ResBlock(96, 128, stride=1),
            ResBlock(128, 192, stride=2),                  # H/8, W/8
            ResBlock(192, out_ch, stride=1),
            ResBlock(out_ch, out_ch, stride=1),
        )
    def forward(self, x):
        return self.net(x)


class TextEncoder(nn.Module):
    def __init__(self, vocab_size, emb_dim=192, hidden=256, out_dim=256, pad_idx=0):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_idx)
        self.gru = nn.GRU(emb_dim, hidden, batch_first=True, bidirectional=True)
        self.proj = nn.Sequential(
            nn.Linear(hidden * 2, out_dim),
            nn.LayerNorm(out_dim),
            nn.SiLU(inplace=True),
        )

    def forward(self, tokens):
        emb = self.embedding(tokens)
        lengths = (tokens != 0).sum(dim=1).clamp(min=1).cpu()
        packed = nn.utils.rnn.pack_padded_sequence(emb, lengths, batch_first=True, enforce_sorted=False)
        _, h = self.gru(packed)
        q = torch.cat([h[-2], h[-1]], dim=1)

        return self.proj(q)


class FiLMBlock(nn.Module):
    def __init__(self, channels=256, q_dim=256):
        super().__init__()
        self.conv1 = ConvBNAct(channels, channels, 3, stride=1)
        self.conv2 = nn.Sequential(
            nn.Conv2d(channels, channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(channels),
        )
        self.to_gamma_beta = nn.Linear(q_dim, channels * 2)
        self.act = nn.SiLU(inplace=True)

    def forward(self, x, q):
        y = self.conv1(x)
        y = self.conv2(y)
        gamma, beta = self.to_gamma_beta(q).chunk(2, dim=1)
        y = y * (1.0 + gamma[:, :, None, None]) + beta[:, :, None, None]

        return self.act(x + y)


def add_coord_channels(x):
    b, _, h, w = x.shape
    yy = torch.linspace(-1, 1, h, device=x.device, dtype=x.dtype).view(1, 1, h, 1).expand(b, 1, h, w)
    xx = torch.linspace(-1, 1, w, device=x.device, dtype=x.dtype).view(1, 1, 1, w).expand(b, 1, h, w)

    return torch.cat([x, xx, yy], dim=1)


class SpatialAttentionPool(nn.Module):
    def __init__(self, channels=256, q_dim=256, att_dim=256, glimpses=2):
        super().__init__()
        self.glimpses = glimpses
        self.v_proj = nn.Conv2d(channels, att_dim, 1)
        self.q_proj = nn.Linear(q_dim, att_dim)
        self.att = nn.Conv2d(att_dim, glimpses, 1)

    def forward(self, v, q):
        b, c, h, w = v.shape
        joint = torch.tanh(self.v_proj(v) + self.q_proj(q)[:, :, None, None])
        logits = self.att(joint).view(b, self.glimpses, h * w)
        att = torch.softmax(logits, dim=-1)
        flat = v.view(b, c, h * w)
        pooled = torch.einsum("bgp,bcp->bgc", att, flat).reshape(b, self.glimpses * c)

        return pooled


class SynVQAFiLM(nn.Module):
    def __init__(self, vocab_size, num_answers, width=256, q_dim=256, film_blocks=4, glimpses=2, dropout=0.2):
        super().__init__()
        self.image = ImageEncoder(out_ch=width)
        self.text = TextEncoder(vocab_size, emb_dim=192, hidden=256, out_dim=q_dim)
        self.coord_proj = ConvBNAct(width + 2, width, kernel=1, padding=0)
        self.film = nn.ModuleList([FiLMBlock(width, q_dim) for _ in range(film_blocks)])
        self.pool = SpatialAttentionPool(width, q_dim, att_dim=width, glimpses=glimpses)
        self.classifier = nn.Sequential(
            nn.Linear(width * glimpses + q_dim, 512),
            nn.LayerNorm(512),
            nn.SiLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(512, 512),
            nn.LayerNorm(512),
            nn.SiLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(512, num_answers),
        )

    def forward(self, img, tokens):
        q = self.text(tokens)
        v = self.image(img)
        v = self.coord_proj(add_coord_channels(v))

        for block in self.film:
            v = block(v, q)

        pooled = self.pool(v, q)

        return self.classifier(torch.cat([pooled, q], dim=1))


def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

tmp_model = SynVQAFiLM(len(vocab), len(idx2ans), dropout=CFG.DROPOUT)
print("Trainable parameters:", f"{count_params(tmp_model)/1e6:.2f}M")
del tmp_model


Trainable parameters: 10.29M


In [28]:
class ModelEMA:
    def __init__(self, model, decay=0.997):
        self.decay = decay
        self.shadow = {}
        self.backup = {}

        for k, v in model.state_dict().items():
            if torch.is_floating_point(v):
                self.shadow[k] = v.detach().clone()

    @torch.no_grad()
    def update(self, model):
        sd = model.state_dict()
        for k, v in sd.items():
            if k in self.shadow:
                self.shadow[k].mul_(self.decay).add_(v.detach(), alpha=1.0 - self.decay)

    @torch.no_grad()
    def apply_shadow(self, model):
        self.backup = {}
        sd = model.state_dict()
        for k, v in self.shadow.items():
            self.backup[k] = sd[k].detach().clone()
            sd[k].copy_(v)

    @torch.no_grad()
    def restore(self, model):
        sd = model.state_dict()
        for k, v in self.backup.items():
            sd[k].copy_(v)

        self.backup = {}


def build_optimizer(model):
    decay, no_decay = [], []
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if p.ndim == 1 or name.endswith(".bias") or "norm" in name.lower() or "bn" in name.lower():
            no_decay.append(p)
        else:
            decay.append(p)

    return torch.optim.AdamW(
        [
            {"params": decay, "weight_decay": CFG.WEIGHT_DECAY},
            {"params": no_decay, "weight_decay": 0.0},
        ],
        lr=CFG.LR,
        betas=(0.9, 0.98),
        eps=1e-8,
    )


def amp_context(device):
    if CFG.AMP and device.type == "cuda":
        return torch.cuda.amp.autocast()

    return nullcontext()


def make_loss():
    try:
        return nn.CrossEntropyLoss(label_smoothing=CFG.LABEL_SMOOTHING)
    except TypeError:
        return nn.CrossEntropyLoss()


@torch.no_grad()
def evaluate(model, loader, device, answer_groups):
    model.eval()
    total = 0
    correct_raw = 0
    correct_masked = 0
    by_domain = collections.defaultdict(lambda: [0, 0])

    for img, tokens, y, qids, questions in tqdm(loader, desc="val", leave=False):
        img = img.to(device, non_blocking=True)
        tokens = tokens.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        logits = model(img, tokens).float()

        raw_pred = logits.argmax(dim=1)
        masked_logits = apply_answer_mask(logits, questions, answer_groups, CFG.ANSWER_MASK_MODE)
        masked_pred = masked_logits.argmax(dim=1)

        correct_raw += (raw_pred == y).sum().item()
        correct_masked += (masked_pred == y).sum().item()
        total += y.numel()

        for qi, pred_i, target_i in zip(qids, masked_pred.cpu().tolist(), y.cpu().tolist()):
            key = "closure" if "closure" in str(qi).lower() else ("clevr" if "clevr" in str(qi).lower() else "unknown")
            by_domain[key][0] += int(pred_i == target_i)
            by_domain[key][1] += 1

    raw_acc = correct_raw / max(total, 1)
    masked_acc = correct_masked / max(total, 1)
    domain_acc = {k: v[0] / max(v[1], 1) for k, v in by_domain.items()}

    return raw_acc, masked_acc, domain_acc


def save_checkpoint(path, model, epoch, score, cfg_extra=None, optimizer=None, scheduler=None, scaler=None, ema=None, best_score=None):
    payload = {
        "model": model.state_dict(),
        "epoch": int(epoch),
        "score": float(score),
        "best_score": float(best_score if best_score is not None else score),
        "vocab": vocab,
        "idx2ans": idx2ans,
        "cfg": {
            "img_h": CFG.IMG_H,
            "img_w": CFG.IMG_W,
            "max_len": CFG.MAX_LEN,
            "answer_mask_mode": CFG.ANSWER_MASK_MODE,
            **(cfg_extra or {}),
        }
    }

    if optimizer is not None:
        payload["optimizer"] = optimizer.state_dict()
    if scheduler is not None:
        payload["scheduler"] = scheduler.state_dict()
    if scaler is not None:
        payload["scaler"] = scaler.state_dict()
    if ema is not None:
        payload["ema_shadow"] = {k: v.detach().clone() for k, v in ema.shadow.items()}
        payload["ema_decay"] = float(ema.decay)

    torch.save(payload, path)


def load_last_checkpoint_if_exists(last_path, model, optimizer, scheduler, scaler, ema, device):
    if not Path(last_path).exists():
        return 1, -1.0

    ckpt = torch.load(last_path, map_location=device)
    model.load_state_dict(ckpt["model"], strict=True)

    if "optimizer" in ckpt:
        optimizer.load_state_dict(ckpt["optimizer"])
    if "scheduler" in ckpt:
        scheduler.load_state_dict(ckpt["scheduler"])
    if "scaler" in ckpt:
        scaler.load_state_dict(ckpt["scaler"])
    if "ema_shadow" in ckpt and isinstance(ckpt["ema_shadow"], dict):
        ema.shadow = ckpt["ema_shadow"]

    start_epoch = int(ckpt.get("epoch", 0)) + 1
    best_score = float(ckpt.get("best_score", ckpt.get("score", -1.0)))
    return start_epoch, best_score


def train_one_seed(seed: int, train_items, val_items):
    seed_everything(seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    train_ds = SynVQADataset(train_items, TRAIN_IMG_DIR, vocab, ans2idx, train=True, return_raw=False)
    val_ds = SynVQADataset(val_items, TRAIN_IMG_DIR, vocab, ans2idx, train=True, return_raw=True) if val_items else None
    train_loader = make_loader(train_ds, CFG.BATCH_SIZE, shuffle=True)
    val_loader = make_loader(val_ds, CFG.BATCH_SIZE * 2, shuffle=False) if val_ds is not None else None

    model = SynVQAFiLM(len(vocab), len(idx2ans), dropout=CFG.DROPOUT).to(device)

    if CFG.COMPILE and hasattr(torch, "compile"):
        model = torch.compile(model)

    optimizer = build_optimizer(model)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer,
        max_lr=CFG.LR,
        epochs=CFG.EPOCHS,
        steps_per_epoch=max(1, len(train_loader)),
        pct_start=0.12,
        div_factor=15,
        final_div_factor=100,
    )
    loss_fn = make_loss()
    scaler = torch.cuda.amp.GradScaler(enabled=(CFG.AMP and device.type == "cuda"))
    ema = ModelEMA(model, decay=0.997)

    answer_groups_device = {k: v.to(device) for k, v in answer_groups_cpu.items()}
    best_score = -1.0
    best_path = CFG.MODEL_DIR / f"synvqa_film_seed{seed}.pth"
    last_path = CFG.MODEL_DIR / f"synvqa_film_seed{seed}.last.pth"

    start_epoch, loaded_best_score = load_last_checkpoint_if_exists(
        last_path=last_path,
        model=model,
        optimizer=optimizer,
        scheduler=scheduler,
        scaler=scaler,
        ema=ema,
        device=device,
    )
    best_score = max(best_score, loaded_best_score)

    print(f"\n========== Seed {seed} | train={len(train_ds)} | val={len(val_ds) if val_ds else 0} ==========")
    if start_epoch > 1:
        print(f"Resuming seed {seed} from epoch {start_epoch} using {last_path}")
    if start_epoch > CFG.EPOCHS:
        print(f"Seed {seed} already completed up to epoch {start_epoch - 1} (CFG.EPOCHS={CFG.EPOCHS}).")
        return best_path, best_score

    for epoch in range(start_epoch, CFG.EPOCHS + 1):
        model.train()
        running_loss = 0.0
        running_correct = 0
        running_total = 0
        pbar = tqdm(train_loader, desc=f"seed {seed} epoch {epoch}/{CFG.EPOCHS}")

        for img, tokens, y in pbar:
            img = img.to(device, non_blocking=True)
            tokens = tokens.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            with amp_context(device):
                logits = model(img, tokens)
                loss = loss_fn(logits, y)

            scaler.scale(loss).backward()

            if CFG.GRAD_CLIP is not None and CFG.GRAD_CLIP > 0:
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), CFG.GRAD_CLIP)

            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            ema.update(model)

            running_loss += loss.item() * y.size(0)
            running_correct += (logits.detach().argmax(dim=1) == y).sum().item()
            running_total += y.size(0)
            pbar.set_postfix(
                loss=running_loss / max(running_total, 1),
                acc=running_correct / max(running_total, 1),
                lr=optimizer.param_groups[0]["lr"],
            )

        train_loss = running_loss / max(running_total, 1)
        train_acc = running_correct / max(running_total, 1)

        if val_loader is not None:
            ema.apply_shadow(model)
            raw_acc, masked_acc, domain_acc = evaluate(model, val_loader, device, answer_groups_device)
            ema.restore(model)
            score = masked_acc
            print(f"Epoch {epoch:02d}: train_loss={train_loss:.4f} train_acc={train_acc:.4f} val_raw={raw_acc:.4f} val_masked={masked_acc:.4f} domains={domain_acc}")

            if score > best_score:
                best_score = score
                ema.apply_shadow(model)
                save_checkpoint(best_path, model, epoch, best_score, {"seed": seed, "kind": "best"})
                ema.restore(model)
                print("  saved best ->", best_path)
        else:
            # No validation: keep latest EMA checkpoint as best proxy.
            best_score = train_acc
            ema.apply_shadow(model)
            save_checkpoint(best_path, model, epoch, best_score, {"seed": seed, "kind": "best"})
            ema.restore(model)
            print(f"Epoch {epoch:02d}: train_loss={train_loss:.4f} train_acc={train_acc:.4f} saved latest as best")

        # Save resumable last checkpoint at end of every completed epoch.
        # If interruption happens mid-next-epoch, rerun that interrupted epoch on resume.
        save_checkpoint(
            last_path,
            model,
            epoch,
            score=best_score,
            cfg_extra={"seed": seed, "kind": "last"},
            optimizer=optimizer,
            scheduler=scheduler,
            scaler=scaler,
            ema=ema,
            best_score=best_score,
        )

    if not best_path.exists():
        ema.apply_shadow(model)
        save_checkpoint(best_path, model, CFG.EPOCHS, best_score, {"seed": seed, "kind": "best_fallback"})
        ema.restore(model)

    return best_path, best_score




In [29]:
# Each seed uses a different image-level validation split. The final prediction averages all seed logits.
trained_paths = []
val_scores = []
start = time.time()

for seed in CFG.SEEDS:
    train_items, val_items = split_by_image(train_items_all, seed, CFG.VAL_RATIO)
    path, score = train_one_seed(seed, train_items, val_items)
    trained_paths.append(path)
    val_scores.append(score)

print("\nTrained checkpoints:")

for p, s in zip(trained_paths, val_scores):
    print(f"  {p} | best masked val acc={s:.4f}")

print(f"Total training time: {(time.time() - start) / 60:.1f} min")


/tmp/ipykernel_2200/4170576431.py:180: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(CFG.AMP and device.type == "cuda"))



========== Seed 3407 | train=664989 | val=35000 ==========
Resuming seed 3407 from epoch 29 using outputs/models/synvqa_film_seed3407.last.pth
Seed 3407 already completed up to epoch 28 (CFG.EPOCHS=28).

========== Seed 2025 | train=664989 | val=35000 ==========
Resuming seed 2025 from epoch 7 using outputs/models/synvqa_film_seed2025.last.pth


seed 2025 epoch 7/28:   0%|          | 0/10391 [00:00<?, ?it/s]/tmp/ipykernel_2200/4170576431.py:58: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  return torch.cuda.amp.autocast()
seed 2025 epoch 7/28: 100%|██████████| 10391/10391 [53:47<00:00,  3.22it/s, acc=0.695, loss=0.767, lr=0.000237]


Epoch 07: train_loss=0.7670 train_acc=0.6949 val_raw=0.7213 val_masked=0.6152 domains={'clevr': 0.6151714285714286}
  saved best -> outputs/models/synvqa_film_seed2025.pth


seed 2025 epoch 8/28:   1%|          | 74/10391 [00:31<1:12:04,  2.39it/s, acc=0.736, loss=0.717, lr=0.000237]


KeyboardInterrupt: 

In [ ]:
def load_model_from_checkpoint(path: Path, device):
    ckpt = torch.load(path, map_location=device)
    model = SynVQAFiLM(len(vocab), len(idx2ans), dropout=CFG.DROPOUT).to(device)
    model.load_state_dict(ckpt["model"], strict=True)
    model.eval()
                  
    return model


@torch.no_grad()
def predict_ensemble(checkpoint_paths):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    test_ds = SynVQADataset(test_items, TEST_IMG_DIR, vocab, ans2idx=None, train=False, return_raw=True)
    test_loader = make_loader(test_ds, CFG.BATCH_SIZE * 2, shuffle=False)

    models = [load_model_from_checkpoint(Path(p), device) for p in checkpoint_paths]
    answer_groups_device = {k: v.to(device) for k, v in answer_groups_cpu.items()}
    rows = []

    for img, tokens, qids, questions in tqdm(test_loader, desc="test inference"):
        img = img.to(device, non_blocking=True)
        tokens = tokens.to(device, non_blocking=True)

        avg_logits = None

        for model in models:
            logits = model(img, tokens).float()
            avg_logits = logits if avg_logits is None else avg_logits + logits
        avg_logits = avg_logits / len(models)
        avg_logits = apply_answer_mask(avg_logits, questions, answer_groups_device, CFG.ANSWER_MASK_MODE)
        pred = avg_logits.argmax(dim=1).cpu().tolist()

        for qid, pi in zip(qids, pred):
            rows.append([qid, idx2ans[pi]])

    return rows

rows = predict_ensemble(trained_paths)

with open(CFG.SUBMISSION_CSV, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["qid", "answer"])
    writer.writerows(rows)

with zipfile.ZipFile(CFG.SUBMISSION_ZIP, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write(CFG.SUBMISSION_CSV, arcname="submission.csv")

print("Saved:", CFG.SUBMISSION_CSV)
print("Saved:", CFG.SUBMISSION_ZIP)
print("Rows:", len(rows))
print("Preview:")

for r in rows[:10]:
    print(r)


test inference: 100%|██████████| 1781/1781 [09:06<00:00,  3.26it/s]


Saved: outputs/submission.csv
Saved: outputs/submission.zip
Rows: 227899
Preview:
['clevr_val_0', 'no']
['clevr_val_1', 'yes']
['clevr_val_2', 'metal']
['clevr_val_3', 'yes']
['clevr_val_4', 'no']
['clevr_val_5', '2']
['clevr_val_6', '2']
['clevr_val_7', 'purple']
['clevr_val_8', 'rubber']
['clevr_val_9', 'no']
